In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

In [2]:
pip install openai

Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)


In [7]:
# 스페인어 필사
with open("data/spanish_audio.mp3", "rb") as audio_file:
    response_es = client.audio.transcriptions.create(
        file = audio_file,
        model = 'whisper-1',
        response_format="text",
        language="es" # 입력 음성 언어를 지정
    )
print(response_es)

¿Qué crees que es la inteligencia artificial?



In [8]:
# 1. 음성 파일 열기
audio_file = open('data/speech.mp3', 'rb')

# 2. whisper API에 음성 필사 요청
response = client.audio.transcriptions.create(
    file=audio_file,
    model = 'whisper-1',
    response_format="srt" # 텍스트만 반환
)

# 3. 변환된 텍스트 출력
print(response)
audio_file.close()

1
00:00:00,000 --> 00:00:04,000
OpenAI Whisper is an advanced speech recognition model.





In [10]:
with open('data/ch5_shimmer_ko_hd.wav','rb') as audio_file:
    transcript_json = client.audio.transcriptions.create(
        file = audio_file,
        model = "whisper-1",
        response_format = "verbose_json",
        language="ko"
    )
transcript_json

TranscriptionVerbose(duration=3.4600000381469727, language='korean', text='안녕하세요. 저는 OpenAPI로 생성된 한국어 음성입니다. 한국어로 자동인식되어 읽고 있어요.', segments=[TranscriptionSegment(id=0, avg_logprob=-0.26857027411460876, compression_ratio=1.0521739721298218, end=4.0, no_speech_prob=0.0068762884475290775, seek=0, start=0.0, temperature=0.0, text=' 안녕하세요. 저는 OpenAPI로 생성된 한국어 음성입니다. 한국어로 자동인식되어 읽고 있어요.', tokens=[50364, 19289, 13, 10551, 7238, 4715, 40, 12888, 6439, 8631, 27930, 21045, 3103, 15179, 8631, 7416, 13, 21045, 6540, 1955, 15905, 8309, 4215, 10436, 22826, 3103, 43302, 1313, 12654, 13, 50564])], usage=Usage(duration=None, type='duration', seconds=4), words=None, task='transcribe')

In [11]:
segment = transcript_json.segments[0]
print("text구간 {}=>{}".format(segment.start, segment.end))
print(segment.text)

text구간 0.0=>4.0
 안녕하세요. 저는 OpenAPI로 생성된 한국어 음성입니다. 한국어로 자동인식되어 읽고 있어요.


In [14]:
import sounddevice as sd
from scipy.io.wavfile import write # mp3는 스트리밍이 안 되어, wav로 스트리밍

fs = 16000 # 샘플레이트 16kHz
seconds = 5 # 음성 녹음 길이 5초
print("지금부터 5초간 녹음합니다")

recording = sd.rec(int(seconds*fs), samplerate=fs, channels=1, dtype="int16")
sd.wait()
file_name = 'data/ch6_live_input.wav'
write(file_name, fs, recording)
print('녹음된 음성이 저장되었습니다(크기:{}byte)'.format(len(recording)))

# from IPython.display import Audio
# Audio(filename=file_name, autoplay=True)

# 저장된 녹음 파일을 whisper API로 전송 text로 받기
with open(file_name, "rb") as audio_file:
    live_transcript = client.audio.transcriptions.create(
        file = audio_file,
        model = 'whisper-1',
        response_format="text",
        language = "ko"
    )
print("실시간 녹음 변환 텍스트 :", live_transcript)

# 파일 삭제
import os
try:
    os.remove(file_name)
    print(f"{file_name} 파일이 삭제되었습니다.")
except Exception as e:
    print(f"파일 삭제 중 오류 발생 : {e}")

지금부터 5초간 녹음합니다
녹음된 음성이 저장되었습니다(크기:80000byte)
실시간 녹음 변환 텍스트 : 

data/ch6_live_input.wav 파일이 삭제되었습니다.


In [15]:
# 퀴즈

In [16]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import time

In [17]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [21]:
# 1. 텍스트 파일 읽기
with open("data/ch06_quiz.txt", "r", encoding="utf-8") as file:
    original_text = file.read()

# 2. 텍스트 요약 (70자 이내)
summary_response = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {"role": "system", "content": "다음 텍스트를 50자 이내로 요약해 주고 너는 해적말투로 해야해."},
        {"role": "user", "content": original_text}
    ])
summary_text = summary_response.choices[0].message.content.strip()

In [22]:
print(summary_text)

아 Harold! AI는 기계 지능으로, 자연언어·비전 등 무서운 힘을 갖췄다네!


In [24]:

speech_response = client.audio.speech.create(
    model="tts-1",
    voice="nova",  # 지정된 보이스
    input=summary_text
)

# 4. MP3 파일로 저장
with open("data/ch06_quiz.mp3", "wb") as f:
    f.write(speech_response.content)

print("ch06_quiz.mp3 파일이 성공적으로 생성되었습니다.")


ch06_quiz.mp3 파일이 성공적으로 생성되었습니다.
